# Causality

This notebook tests whether a causal question can be associated with the one independent edit that changes its two-hop answer. It starts with forward-only `A -> B -> C` chains from UniEdit, including valid nested single paths from double-path records. The edit may occupy hop 1 (position 0) or hop 2 (position 1), and every result reports those positions separately.

Local mode asks whether the responsible edit scores above controlled negative edits. Global mode asks whether it can be retrieved from increasingly large edit banks. Only the causal prompt and candidate `{subject, prompt, answer}` are visible to scorers; hop IDs and dependency labels remain hidden evaluation metadata. These experiments measure association and retrieval, not causal success by themselves.

## Environment

In [ ]:
from contextlib import redirect_stderr

import gc
import hashlib
from io import StringIO
import json
import math
from pathlib import Path
import re
import sys
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "causality"
DATASET_SPLIT = "data"
LOCAL_DATASET_PATH = PROJECT_ROOT / "data/datasets/uniedit/final"
OUTPUT_DIR = PROJECT_ROOT / "data/causality"
EMBEDDING_MODEL_REPO = "BAAI/bge-m3"
SCORER_ORDER = (
    "bm25_prompt",
    "bge_prompt",
    "bge_subject_max",
    "bge_composed",
)
SCORER_LABELS = {
    "bm25_prompt": "BM25: edit prompt",
    "bge_prompt": "BGE-M3: edit prompt",
    "bge_subject_max": "BGE-M3: max(prompt, subject)",
    "bge_composed": "BGE-M3: subject + edit + answer",
}

print(f"local_dataset={LOCAL_DATASET_PATH}")
print(f"output_dir={OUTPUT_DIR}")

## Dataset Contract

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(str(text).split()).casefold()


def prompt_key(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", str(text)).casefold()
    normalized = "".join(
        character
        if character.isalnum() or unicodedata.category(character).startswith("S")
        else " "
        for character in normalized
    )
    return " ".join(normalized.split())


def answer_keys(values: list[str]) -> set[str]:
    keys = set()
    for value in values:
        normalized = unicodedata.normalize("NFKC", str(value)).casefold()
        words = tuple(re.findall(r"\w+", normalized))
        if words:
            keys.add("".join(words))
            if len(words) > 1 and words[0] in {"a", "an", "the"}:
                keys.add("".join(words[1:]))
        compact = "".join(
            character
            for character in normalized
            if character.isalnum()
            or unicodedata.category(character).startswith("S")
        )
        if compact:
            keys.add(compact)
    return keys


def owner_id(edit: dict) -> str:
    return f"{edit['subject']['id']}|{edit['relation']['id']}"


def runtime_edit(edit: dict) -> dict[str, str]:
    return {
        "subject": edit["subject"]["str"],
        "prompt": edit["prompt"],
        "answer": edit["answers"][0],
    }


def runtime_query(causality: dict) -> str:
    return causality["prompt"]


def load_causality_dataset(dataset_source: str):
    if dataset_source == "local":
        dataset = load_from_disk(str(LOCAL_DATASET_PATH))
        return dataset["causality"]
    if dataset_source == "huggingface":
        return load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    raise ValueError("dataset_source must be 'local' or 'huggingface'.")


def source_record_count(dataset, record_limit: int | None) -> int:
    if record_limit is None:
        return len(dataset)
    if int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    return min(int(record_limit), len(dataset))


def build_causality_records(dataset, record_limit: int | None) -> list[dict]:
    records = []
    seen_owners = set()
    seen_edit_prompts = set()
    seen_queries = set()
    for source_index in range(source_record_count(dataset, record_limit)):
        source = dataset[source_index]
        edit = source["edit"]
        current_owner = owner_id(edit)
        edit_prompt_key = prompt_key(edit["prompt"])
        if current_owner in seen_owners:
            raise ValueError(f"Duplicate edit owner at row {source_index}: {current_owner}")
        if edit_prompt_key in seen_edit_prompts:
            raise ValueError(f"Duplicate canonical edit prompt at row {source_index}.")
        if not edit["answers"] or not edit["subject"]["str"]:
            raise ValueError(f"Incomplete edit at row {source_index}.")
        seen_owners.add(current_owner)
        seen_edit_prompts.add(edit_prompt_key)

        queries = []
        for query_index, causal in enumerate(source["causality"]):
            query_prompt = runtime_query(causal)
            query_prompt_key = prompt_key(query_prompt)
            if not query_prompt_key or query_prompt_key in seen_queries:
                raise ValueError(f"Missing or repeated causal prompt at row {source_index}.")
            if query_prompt_key in seen_edit_prompts:
                raise ValueError(f"Causal prompt collides with an edit prompt at row {source_index}.")
            if not causal["answers"] or len(causal["hops"]) != 2:
                raise ValueError(f"Incomplete causal evidence at row {source_index}.")
            position = int(causal["source_edit_position"])
            if position not in {0, 1}:
                raise ValueError(f"Invalid source edit position at row {source_index}.")
            hops = causal["hops"]
            if sum(bool(hop["is_source_edit"]) for hop in hops) != 1:
                raise ValueError(f"Causal query has multiple source edits at row {source_index}.")
            if not hops[position]["is_source_edit"]:
                raise ValueError(f"Source position disagrees with hop labels at row {source_index}.")
            if any(
                hop["direction"] != "forward" or not hop["single_value"]
                for hop in hops
            ):
                raise ValueError(f"Non-forward or multi-valued hop at row {source_index}.")
            if hops[0]["tail"]["id"] != hops[1]["head"]["id"]:
                raise ValueError(f"Broken two-hop entity continuity at row {source_index}.")
            if (
                hops[position]["head"]["id"] != edit["subject"]["id"]
                or hops[position]["relation"]["id"] != edit["relation"]["id"]
            ):
                raise ValueError(f"Source hop disagrees with its edit at row {source_index}.")
            edit_answer_keys = answer_keys(list(edit["answers"]))
            if not edit_answer_keys & answer_keys(list(hops[position]["answers"])):
                raise ValueError(f"Source hop answer disagrees with edit at row {source_index}.")
            expected_dependency = [{
                "subject_id": edit["subject"]["id"],
                "relation_id": edit["relation"]["id"],
            }]
            if list(causal["dependency_edit_keys"]) != expected_dependency:
                raise ValueError(f"Causal query is not single-dependency at row {source_index}.")
            causal_answer_keys = answer_keys(list(causal["answers"]))
            if position == 0 and edit_answer_keys & causal_answer_keys:
                raise ValueError(f"Position-0 query can bypass hop two at row {source_index}.")
            if position == 1 and not edit_answer_keys & causal_answer_keys:
                raise ValueError(f"Position-1 query answer disagrees with edit at row {source_index}.")
            seen_queries.add(query_prompt_key)
            query_digest = hashlib.sha256(query_prompt_key.encode("utf-8")).hexdigest()[:12]
            queries.append({
                "query_id": f"{current_owner}|{query_index}|{query_digest}",
                "prompt": query_prompt,
                "answers": list(causal["answers"]),
                "source_edit_position": position,
                "origin": causal["origin"],
                "hidden_hops": list(hops),
            })
        if not queries:
            raise ValueError(f"Edit row {source_index} has no causal query.")
        records.append({
            "source_index": source_index,
            "owner_id": current_owner,
            "edit": runtime_edit(edit),
            "edit_answers": list(edit["answers"]),
            "queries": queries,
        })

    all_edit_keys = {prompt_key(record["edit"]["prompt"]) for record in records}
    all_query_keys = {
        prompt_key(query["prompt"])
        for record in records
        for query in record["queries"]
    }
    if all_edit_keys & all_query_keys:
        raise ValueError("Final causal prompts overlap canonical edit prompts.")
    return records


def causality_prompt_fingerprint(records: list[dict]) -> str:
    digest = hashlib.sha256()
    for record in records:
        payload = {
            "owner_id": record["owner_id"],
            "edit": record["edit"],
            "queries": [
                {
                    "prompt": query["prompt"],
                    "answers": query["answers"],
                    "source_edit_position": query["source_edit_position"],
                    "origin": query["origin"],
                }
                for query in record["queries"]
            ],
        }
        encoded = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def causality_dataset_metadata(dataset, records: list[dict], dataset_source: str) -> dict:
    metadata = {
        "source": dataset_source,
        "rows": len(records),
        "queries": sum(len(record["queries"]) for record in records),
        "fingerprint": getattr(dataset, "_fingerprint", None),
        "prompt_fingerprint": causality_prompt_fingerprint(records),
        "policy": {
            "independent_edits": True,
            "forward_only": True,
            "hop_count": 2,
            "outer_double_questions": False,
            "nested_single_paths": True,
        },
    }
    if dataset_source == "local":
        metadata["dataset_id"] = "uniedit/final:causality"
    else:
        metadata.update({
            "repo": DATASET_REPO,
            "config": DATASET_CONFIG,
            "split": DATASET_SPLIT,
        })
    return metadata

## Scorers

In [ ]:
def embedding_device() -> str:
    return f"cuda:{EMBEDDING_GPU_ID}" if torch.cuda.is_available() else "cpu"


def embedding_compute_dtype() -> str:
    return "float16" if EMBEDDING_USE_FP16 and torch.cuda.is_available() else "float32"


def embedding_precision_mode() -> str:
    return "fp16" if embedding_compute_dtype() == "float16" else "fp32"


def load_embedding_model():
    from FlagEmbedding import BGEM3FlagModel

    model = BGEM3FlagModel(
        EMBEDDING_MODEL_REPO,
        use_fp16=embedding_compute_dtype() == "float16",
        devices=embedding_device(),
    )
    print(
        f"loaded_embedding_model={EMBEDDING_MODEL_REPO} "
        f"device={embedding_device()} compute_dtype={embedding_compute_dtype()}"
    )
    return model


def encode_prompts(model, prompts: list[str]) -> np.ndarray:
    if not prompts:
        raise ValueError("Cannot encode an empty prompt list.")
    with redirect_stderr(StringIO()):
        encoded = model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=EMBEDDING_MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
    embeddings = np.asarray(encoded["dense_vecs"], dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(prompts):
        raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")
    return embeddings


def release_embedding_model(model) -> None:
    underlying = getattr(model, "model", None)
    if underlying is not None and hasattr(underlying, "to"):
        underlying.to("cpu")
    model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def lexical_tokens(text: str) -> tuple[str, ...]:
    return tuple(re.findall(r"\w+", normalize_text(text)))


class Bm25Index:
    def __init__(self, documents: list[str], k1: float = 1.5, b: float = 0.75):
        if not documents:
            raise ValueError("BM25 needs at least one document.")
        self.k1 = float(k1)
        self.b = float(b)
        self.document_count = len(documents)
        self.document_lengths = np.asarray(
            [len(lexical_tokens(document)) for document in documents],
            dtype=np.float32,
        )
        self.average_length = max(float(self.document_lengths.mean()), 1.0)
        document_frequencies = {}
        postings = {}
        for document_index, document in enumerate(documents):
            frequencies = {}
            for token in lexical_tokens(document):
                frequencies[token] = frequencies.get(token, 0) + 1
            for token, frequency in frequencies.items():
                postings.setdefault(token, []).append((document_index, frequency))
                document_frequencies[token] = document_frequencies.get(token, 0) + 1
        self.postings = postings
        self.idf = {
            token: math.log(
                1.0
                + (self.document_count - frequency + 0.5)
                / (frequency + 0.5)
            )
            for token, frequency in document_frequencies.items()
        }

    def score(self, query: str) -> np.ndarray:
        scores = np.zeros(self.document_count, dtype=np.float32)
        for token in set(lexical_tokens(query)):
            if token not in self.postings:
                continue
            idf = self.idf[token]
            for document_index, frequency in self.postings[token]:
                length = self.document_lengths[document_index]
                denominator = frequency + self.k1 * (
                    1.0 - self.b + self.b * length / self.average_length
                )
                scores[document_index] += idf * (
                    frequency * (self.k1 + 1.0) / denominator
                )
        return scores


def composed_edit_text(edit: dict[str, str]) -> str:
    return (
        f"Subject: {edit['subject']}\n"
        f"Edit: {edit['prompt']}\n"
        f"Answer: {edit['answer']}"
    )

## Deterministic Cohorts

In [ ]:
def seeded_rng(random_seed: int, *parts) -> np.random.Generator:
    payload = "|".join(str(part) for part in (random_seed, *parts)).encode("utf-8")
    seed = int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")
    return np.random.default_rng(seed)


def owner_is_calibration(owner: str, random_seed: int, fraction: float) -> bool:
    digest = hashlib.sha256(f"{random_seed}|split|{owner}".encode("utf-8")).digest()
    value = int.from_bytes(digest[:8], "big") / 2**64
    return value < fraction


def flatten_queries(records: list[dict]) -> list[dict]:
    queries = []
    for owner_index, record in enumerate(records):
        for query in record["queries"]:
            queries.append({
                **query,
                "owner_index": owner_index,
                "owner_id": record["owner_id"],
                "expected_edit": record["edit"],
            })
    return queries


def sample_queries(
    queries: list[dict],
    limit: int | None,
    random_seed: int,
    label: str,
) -> list[dict]:
    if limit is None or len(queries) <= int(limit):
        return list(queries)
    if int(limit) <= 0:
        raise ValueError("Query limits must be positive when set.")
    rng = seeded_rng(random_seed, label)
    indices = np.sort(rng.choice(len(queries), size=int(limit), replace=False))
    return [queries[int(index)] for index in indices]


def top_lexical_negatives(
    scores: np.ndarray,
    positive_index: int,
    limit: int,
) -> list[int]:
    candidate_count = min(len(scores) - 1, max(limit * 4, 64))
    if candidate_count <= 0:
        return []
    working = scores.copy()
    working[positive_index] = -np.inf
    if candidate_count < len(working):
        indices = np.argpartition(-working, candidate_count - 1)[:candidate_count]
    else:
        indices = np.arange(len(working))
    return [
        int(index)
        for index in indices[np.argsort(-working[indices])]
        if int(index) != positive_index
    ][:limit]


def fill_negative_indices(
    preferred: list[int],
    positive_index: int,
    record_count: int,
    negative_count: int,
    rng: np.random.Generator,
) -> list[int]:
    selected = []
    seen = {positive_index}
    for index in preferred:
        index = int(index)
        if index not in seen:
            seen.add(index)
            selected.append(index)
        if len(selected) == negative_count:
            return selected
    remaining = np.asarray(
        [index for index in range(record_count) if index not in seen],
        dtype=np.int64,
    )
    needed = negative_count - len(selected)
    if needed > len(remaining):
        raise ValueError("Not enough unique edits for the requested negatives.")
    if needed:
        chosen = rng.choice(remaining, size=needed, replace=False)
        selected.extend(int(index) for index in chosen)
    return selected


def build_local_cohorts(
    records: list[dict],
    queries: list[dict],
    bm25: Bm25Index,
    negative_count: int,
    random_seed: int,
    calibration_fraction: float,
) -> list[dict]:
    if negative_count <= 0 or negative_count >= len(records):
        raise ValueError("local_negative_count must be between 1 and bank size - 1.")
    subject_indices = {}
    answer_indices = {}
    for index, record in enumerate(records):
        subject_indices.setdefault(prompt_key(record["edit"]["subject"]), []).append(index)
        answer_indices.setdefault(prompt_key(record["edit"]["answer"]), []).append(index)

    cohorts = []
    for query in tqdm(queries, desc="Build local causal cohorts"):
        positive = int(query["owner_index"])
        lexical_scores = bm25.score(query["prompt"])
        random_rng = seeded_rng(random_seed, "local", "random", query["query_id"])
        random_negatives = fill_negative_indices(
            [], positive, len(records), negative_count, random_rng
        )

        edit = records[positive]["edit"]
        preferred = []
        preferred.extend(subject_indices.get(prompt_key(edit["subject"]), []))
        preferred.extend(answer_indices.get(prompt_key(edit["answer"]), []))
        preferred.extend(top_lexical_negatives(lexical_scores, positive, negative_count))
        challenge_rng = seeded_rng(random_seed, "local", "challenge", query["query_id"])
        challenge_negatives = fill_negative_indices(
            preferred, positive, len(records), negative_count, challenge_rng
        )

        for cohort_name, negatives in (
            ("random", random_negatives),
            ("challenge", challenge_negatives),
        ):
            candidate_indices = [positive, *negatives]
            cohorts.append({
                "cohort": cohort_name,
                "query": query,
                "candidate_indices": candidate_indices,
                "bm25_scores": lexical_scores[candidate_indices],
                "calibration": owner_is_calibration(
                    query["owner_id"], random_seed, calibration_fraction
                ),
            })
    return cohorts


def resolve_global_bank_sizes(
    requested_sizes: list[int | None],
    universe_size: int,
) -> list[tuple[int | None, int]]:
    resolved = []
    seen = set()
    for requested in requested_sizes:
        actual = universe_size if requested is None else int(requested)
        if actual < 2:
            raise ValueError("Every global bank must contain at least two edits.")
        if actual > universe_size:
            print(f"skipped_bank_size={actual} universe_size={universe_size}")
            continue
        key = (requested is None, actual)
        if key not in seen:
            seen.add(key)
            resolved.append((requested, actual))
    if not resolved:
        raise ValueError("No configured global bank size fits the dataset.")
    return sorted(resolved, key=lambda item: item[1])


def build_global_cohorts(
    records: list[dict],
    bank_sizes: list[int | None],
    repeats: int,
    query_owner_count: int,
    query_limit: int | None,
    random_seed: int,
) -> list[dict]:
    if repeats <= 0:
        raise ValueError("global_repeats must be positive.")
    resolved_sizes = resolve_global_bank_sizes(bank_sizes, len(records))
    smallest_bank = resolved_sizes[0][1]
    owner_count = min(int(query_owner_count), smallest_bank)
    if owner_count < 2:
        raise ValueError("global_query_owner_count must retain at least two owners.")
    cohorts = []
    for repeat_index in range(repeats):
        rng = seeded_rng(random_seed, "global", repeat_index)
        permutation = rng.permutation(len(records)).astype(np.int64)
        query_owners = set(int(index) for index in permutation[:owner_count])
        query_pool = [
            query
            for query in flatten_queries(records)
            if int(query["owner_index"]) in query_owners
        ]
        queries = sample_queries(
            query_pool,
            query_limit,
            random_seed,
            f"global-repeat-{repeat_index}",
        )
        cohorts.append({
            "repeat_index": repeat_index,
            "permutation": permutation,
            "query_owner_count": owner_count,
            "queries": queries,
            "bank_sizes": resolved_sizes,
        })
    return cohorts


def cohort_fingerprint(payload) -> str:
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

## Metrics

In [ ]:
def rounded(value: float) -> float:
    return round(float(value), 8)


def rank_from_scores(positive_score: float, negative_scores: np.ndarray) -> int:
    return 1 + int(np.sum(negative_scores >= positive_score))


def local_observation_metrics(observations: list[dict]) -> dict:
    if not observations:
        return {"count": 0}
    auc_values = []
    ranks = []
    gaps = []
    wins = ties = losses = 0
    for observation in observations:
        positive = float(observation["positive_score"])
        negatives = np.asarray(observation["negative_scores"], dtype=np.float32)
        auc_values.append(float(np.mean((positive > negatives) + 0.5 * (positive == negatives))))
        rank = rank_from_scores(positive, negatives)
        ranks.append(rank)
        best_negative = float(negatives.max())
        gap = positive - best_negative
        gaps.append(gap)
        wins += gap > 0
        ties += gap == 0
        losses += gap < 0
    ranks = np.asarray(ranks, dtype=np.float32)
    gaps = np.asarray(gaps, dtype=np.float32)
    return {
        "count": len(observations),
        "pairwise_auc": rounded(np.mean(auc_values)),
        "top1_accuracy": rounded(np.mean(ranks == 1)),
        "mrr": rounded(np.mean(1.0 / ranks)),
        "mean_best_negative_gap": rounded(gaps.mean()),
        "gap_std": rounded(gaps.std()),
        "wins": int(wins),
        "ties": int(ties),
        "losses": int(losses),
    }


def fit_balanced_threshold(positive_scores: np.ndarray, negative_scores: np.ndarray) -> float:
    scores = np.concatenate([positive_scores, negative_scores]).astype(np.float64)
    labels = np.concatenate([
        np.ones(len(positive_scores), dtype=np.int8),
        np.zeros(len(negative_scores), dtype=np.int8),
    ])
    order = np.argsort(-scores, kind="mergesort")
    scores = scores[order]
    labels = labels[order]
    unique_starts = np.r_[0, np.flatnonzero(scores[1:] != scores[:-1]) + 1]
    positives = np.add.reduceat(labels, unique_starts)
    group_sizes = np.diff(np.r_[unique_starts, len(scores)])
    negatives = group_sizes - positives
    true_positives = np.cumsum(positives)
    false_positives = np.cumsum(negatives)
    positive_total = max(int(labels.sum()), 1)
    negative_total = max(int((labels == 0).sum()), 1)
    balanced = 0.5 * (
        true_positives / positive_total
        + 1.0 - false_positives / negative_total
    )
    best = int(np.argmax(balanced))
    return float(scores[unique_starts[best]])


def threshold_metrics(observations: list[dict]) -> dict:
    calibration = [item for item in observations if item["calibration"]]
    evaluation = [item for item in observations if not item["calibration"]]
    if not calibration or not evaluation:
        return {"available": False}
    calibration_positive = np.asarray(
        [item["positive_score"] for item in calibration], dtype=np.float32
    )
    calibration_negative = np.concatenate([
        np.asarray(item["negative_scores"], dtype=np.float32)
        for item in calibration
    ])
    threshold = fit_balanced_threshold(calibration_positive, calibration_negative)
    evaluation_positive = np.asarray(
        [item["positive_score"] for item in evaluation], dtype=np.float32
    )
    evaluation_negative = np.concatenate([
        np.asarray(item["negative_scores"], dtype=np.float32)
        for item in evaluation
    ])
    true_positive = int(np.sum(evaluation_positive >= threshold))
    false_positive = int(np.sum(evaluation_negative >= threshold))
    return {
        "available": True,
        "threshold": rounded(threshold),
        "calibration_queries": len(calibration),
        "evaluation_queries": len(evaluation),
        "evaluation_positive_recall": rounded(true_positive / len(evaluation_positive)),
        "evaluation_negative_false_positive_rate": rounded(
            false_positive / len(evaluation_negative)
        ),
        "evaluation_precision": rounded(
            true_positive / max(true_positive + false_positive, 1)
        ),
    }


def global_metrics(scores: np.ndarray, expected_positions: np.ndarray) -> dict:
    row_indices = np.arange(len(scores))
    correct = scores[row_indices, expected_positions]
    greater_or_equal = scores >= correct[:, None]
    greater_or_equal[row_indices, expected_positions] = False
    ranks = 1 + greater_or_equal.sum(axis=1)
    masked = scores.copy()
    masked[row_indices, expected_positions] = -np.inf
    best_negative = masked.max(axis=1)
    margins = correct - best_negative
    return {
        "count": len(scores),
        "recall_at_1": rounded(np.mean(ranks <= 1)),
        "recall_at_5": rounded(np.mean(ranks <= 5)),
        "recall_at_10": rounded(np.mean(ranks <= 10)),
        "recall_at_20": rounded(np.mean(ranks <= 20)),
        "recall_at_50": rounded(np.mean(ranks <= 50)),
        "mrr": rounded(np.mean(1.0 / ranks)),
        "rank_p50": rounded(np.percentile(ranks, 50)),
        "rank_p95": rounded(np.percentile(ranks, 95)),
        "correct_score_mean": rounded(correct.mean()),
        "correct_margin_mean": rounded(margins.mean()),
        "ranks": ranks,
        "correct_scores": correct,
        "margins": margins,
    }


def serializable_global_metrics(metrics: dict) -> dict:
    return {
        key: value
        for key, value in metrics.items()
        if key not in {"ranks", "correct_scores", "margins"}
    }


def aggregate_metric_dicts(metrics: list[dict]) -> dict:
    if not metrics:
        return {}
    output = {}
    for key in metrics[0]:
        values = [item[key] for item in metrics]
        if isinstance(values[0], (int, float)) and key != "count":
            output[key] = {
                "mean": rounded(np.mean(values)),
                "std": rounded(np.std(values)),
            }
        elif key == "count":
            output[key] = int(round(float(np.mean(values))))
    return output

## Calculation

In [ ]:
def result_output_path(experiment_mode: str) -> Path:
    if experiment_mode not in {"local", "global"}:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    stem = "training-free"
    if RECORD_LIMIT is not None:
        stem += f".limit-{int(RECORD_LIMIT)}"
    return OUTPUT_DIR / experiment_mode / embedding_precision_mode() / f"{stem}.json"


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    staging = path.with_suffix(path.suffix + ".staging")
    with staging.open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)
        file.write("\n")
    staging.replace(path)


def experiment_configuration(experiment_mode: str) -> dict:
    if experiment_mode == "local":
        settings = {
            "negative_count": LOCAL_NEGATIVE_COUNT,
            "query_limit": LOCAL_QUERY_LIMIT,
            "calibration_fraction": LOCAL_CALIBRATION_FRACTION,
            "random_seed": RANDOM_SEED,
            "cohorts": ["random", "challenge"],
        }
    elif experiment_mode == "global":
        settings = {
            "bank_sizes": GLOBAL_BANK_SIZES,
            "repeats": GLOBAL_REPEATS,
            "query_owner_count": GLOBAL_QUERY_OWNER_COUNT,
            "query_limit": GLOBAL_QUERY_LIMIT,
            "random_seed": RANDOM_SEED,
            "bank_sampling": "nested_prefixes_with_fixed_query_owners",
            "bank_universe": "retained_causality_owners",
        }
    else:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    return {
        "mode": experiment_mode,
        "record_limit": RECORD_LIMIT,
        "scorers": list(SCORER_ORDER),
        "settings": settings,
    }


def causality_dataset_metadata_matches(
    saved: dict | None,
    current: dict,
) -> bool:
    if saved is None:
        return False
    identity_fields = (
        "source",
        "rows",
        "queries",
        "prompt_fingerprint",
        "policy",
        "dataset_id",
        "repo",
        "config",
        "split",
    )
    return all(saved.get(field) == current.get(field) for field in identity_fields)


def result_matches_configuration(
    path: Path,
    experiment: dict,
    dataset_metadata: dict,
) -> bool:
    if not path.exists():
        return False
    with path.open(encoding="utf-8") as file:
        result = json.load(file)
    return (
        result.get("embedding_model") == EMBEDDING_MODEL_REPO
        and result.get("use_fp16") == EMBEDDING_USE_FP16
        and result.get("max_length") == EMBEDDING_MAX_LENGTH
        and result.get("experiment") == experiment
        and causality_dataset_metadata_matches(
            result.get("dataset"),
            dataset_metadata,
        )
    )


def scorer_arrays(
    query_embeddings: np.ndarray,
    edit_prompt_embeddings: np.ndarray,
    subject_embeddings: np.ndarray,
    composed_embeddings: np.ndarray,
) -> dict[str, np.ndarray]:
    prompt_scores = query_embeddings @ edit_prompt_embeddings.T
    subject_scores = query_embeddings @ subject_embeddings.T
    return {
        "bge_prompt": prompt_scores,
        "bge_subject_max": np.maximum(prompt_scores, subject_scores),
        "bge_composed": query_embeddings @ composed_embeddings.T,
    }


def local_diagnostics(
    observations: list[dict],
    records: list[dict],
    limit: int,
) -> list[dict]:
    failures = sorted(
        (
            item for item in observations
            if item["positive_score"] <= max(item["negative_scores"])
        ),
        key=lambda item: item["positive_score"] - max(item["negative_scores"]),
    )
    output = []
    for item in failures[:limit]:
        negatives = np.asarray(item["negative_scores"], dtype=np.float32)
        best_position = int(np.argmax(negatives))
        retrieved_index = item["negative_indices"][best_position]
        output.append({
            "query": item["query"]["prompt"],
            "source_edit_position": item["query"]["source_edit_position"],
            "expected_edit": item["query"]["expected_edit"],
            "retrieved_edit": records[retrieved_index]["edit"],
            "positive_score": rounded(item["positive_score"]),
            "retrieved_score": rounded(negatives[best_position]),
            "gap": rounded(item["positive_score"] - negatives[best_position]),
        })
    return output


def run_local_experiment(
    records: list[dict],
    queries: list[dict],
    bm25: Bm25Index,
    query_embedding_by_id: dict[str, np.ndarray],
    edit_prompt_embeddings: np.ndarray,
    subject_embeddings: np.ndarray,
    composed_embeddings: np.ndarray,
) -> dict:
    cohorts = build_local_cohorts(
        records,
        queries,
        bm25,
        LOCAL_NEGATIVE_COUNT,
        RANDOM_SEED,
        LOCAL_CALIBRATION_FRACTION,
    )
    observations = {
        method: {"random": [], "challenge": []}
        for method in SCORER_ORDER
    }
    for cohort in tqdm(cohorts, desc="Score local causal cohorts"):
        query = cohort["query"]
        candidates = cohort["candidate_indices"]
        query_embedding = query_embedding_by_id[query["query_id"]][None, :]
        embedding_scores = scorer_arrays(
            query_embedding,
            edit_prompt_embeddings[candidates],
            subject_embeddings[candidates],
            composed_embeddings[candidates],
        )
        method_scores = {
            "bm25_prompt": np.asarray(cohort["bm25_scores"], dtype=np.float32),
            **{key: value[0] for key, value in embedding_scores.items()},
        }
        for method, scores in method_scores.items():
            observations[method][cohort["cohort"]].append({
                "query": query,
                "positive_score": float(scores[0]),
                "negative_scores": [float(value) for value in scores[1:]],
                "negative_indices": list(candidates[1:]),
                "calibration": cohort["calibration"],
            })

    methods = {}
    for method in SCORER_ORDER:
        methods[method] = {"cohorts": {}}
        for cohort_name in ("random", "challenge"):
            current = observations[method][cohort_name]
            groups = {
                "all": current,
                "position_0": [
                    item for item in current
                    if item["query"]["source_edit_position"] == 0
                ],
                "position_1": [
                    item for item in current
                    if item["query"]["source_edit_position"] == 1
                ],
                "origin_single": [
                    item for item in current
                    if item["query"]["origin"] == "single"
                ],
                "origin_nested": [
                    item for item in current
                    if item["query"]["origin"] == "double.single_path1"
                ],
            }
            methods[method]["cohorts"][cohort_name] = {
                "metrics": {
                    group_name: local_observation_metrics(group)
                    for group_name, group in groups.items()
                },
                "threshold": threshold_metrics(current),
                "diagnostics": local_diagnostics(
                    [item for item in current if not item["calibration"]],
                    records,
                    DIAGNOSTIC_LIMIT,
                ),
            }
    cohort_payload = [
        {
            "query_id": item["query"]["query_id"],
            "cohort": item["cohort"],
            "candidate_indices": item["candidate_indices"],
            "calibration": item["calibration"],
        }
        for item in cohorts
    ]
    return {
        "query_count": len(queries),
        "cohort_fingerprint": cohort_fingerprint(cohort_payload),
        "methods": methods,
    }


def global_failure_examples(
    scores: np.ndarray,
    expected_positions: np.ndarray,
    queries: list[dict],
    bank_indices: np.ndarray,
    records: list[dict],
    limit: int,
) -> list[dict]:
    top_positions = np.argmax(scores, axis=1)
    failures = np.flatnonzero(top_positions != expected_positions)
    margins = scores[np.arange(len(scores)), expected_positions] - scores[
        np.arange(len(scores)), top_positions
    ]
    failures = failures[np.argsort(margins[failures])]
    output = []
    for query_position in failures[:limit]:
        retrieved_position = int(top_positions[query_position])
        output.append({
            "query": queries[int(query_position)]["prompt"],
            "source_edit_position": queries[int(query_position)]["source_edit_position"],
            "expected_edit": queries[int(query_position)]["expected_edit"],
            "retrieved_edit": records[int(bank_indices[retrieved_position])]["edit"],
            "expected_score": rounded(
                scores[int(query_position), int(expected_positions[query_position])]
            ),
            "retrieved_score": rounded(scores[int(query_position), retrieved_position]),
            "gap": rounded(margins[int(query_position)]),
        })
    return output


def run_global_experiment(
    records: list[dict],
    bm25: Bm25Index,
    cohorts: list[dict],
    query_embedding_by_id: dict[str, np.ndarray],
    edit_prompt_embeddings: np.ndarray,
    subject_embeddings: np.ndarray,
    composed_embeddings: np.ndarray,
) -> dict:
    scales = {}
    cohort_payload = []
    for cohort in tqdm(cohorts, desc="Run global causal retrieval"):
        queries = cohort["queries"]
        query_embeddings = np.stack([
            query_embedding_by_id[query["query_id"]]
            for query in queries
        ])
        bm25_full = np.stack([bm25.score(query["prompt"]) for query in queries])
        for requested_size, bank_size in cohort["bank_sizes"]:
            bank_indices = cohort["permutation"][:bank_size]
            position_by_owner = {
                int(owner_index): position
                for position, owner_index in enumerate(bank_indices)
            }
            expected_positions = np.asarray([
                position_by_owner[int(query["owner_index"])]
                for query in queries
            ], dtype=np.int64)
            embedding_scores = scorer_arrays(
                query_embeddings,
                edit_prompt_embeddings[bank_indices],
                subject_embeddings[bank_indices],
                composed_embeddings[bank_indices],
            )
            method_scores = {
                "bm25_prompt": bm25_full[:, bank_indices],
                **embedding_scores,
            }
            scale_key = "all" if requested_size is None else str(requested_size)
            scale = scales.setdefault(scale_key, {
                "requested_bank_size": requested_size,
                "bank_size": bank_size,
                "methods": {
                    method: {
                        "repeats": [],
                        "positions": {"0": [], "1": []},
                        "origins": {"single": [], "double.single_path1": []},
                    }
                    for method in SCORER_ORDER
                },
                "diagnostics": {},
            })
            for method, scores in method_scores.items():
                metrics = global_metrics(scores, expected_positions)
                scale["methods"][method]["repeats"].append(
                    serializable_global_metrics(metrics)
                )
                for position in (0, 1):
                    mask = np.asarray([
                        query["source_edit_position"] == position
                        for query in queries
                    ])
                    if mask.any():
                        position_metrics = global_metrics(
                            scores[mask], expected_positions[mask]
                        )
                        scale["methods"][method]["positions"][str(position)].append(
                            serializable_global_metrics(position_metrics)
                        )
                for origin in ("single", "double.single_path1"):
                    mask = np.asarray([
                        query["origin"] == origin
                        for query in queries
                    ])
                    if mask.any():
                        origin_metrics = global_metrics(
                            scores[mask], expected_positions[mask]
                        )
                        scale["methods"][method]["origins"][origin].append(
                            serializable_global_metrics(origin_metrics)
                        )
                if bank_size == cohort["bank_sizes"][-1][1]:
                    examples = global_failure_examples(
                        scores,
                        expected_positions,
                        queries,
                        bank_indices,
                        records,
                        DIAGNOSTIC_LIMIT,
                    )
                    scale["diagnostics"].setdefault(method, []).extend(examples[:1])
            cohort_payload.append({
                "repeat": cohort["repeat_index"],
                "requested_bank_size": requested_size,
                "bank_indices": [int(value) for value in bank_indices],
                "query_ids": [query["query_id"] for query in queries],
            })

    output_scales = []
    for scale in sorted(scales.values(), key=lambda item: item["bank_size"]):
        methods = {}
        for method in SCORER_ORDER:
            method_data = scale["methods"][method]
            methods[method] = {
                "aggregate": aggregate_metric_dicts(method_data["repeats"]),
                "positions": {
                    position: aggregate_metric_dicts(repeats)
                    for position, repeats in method_data["positions"].items()
                    if repeats
                },
                "origins": {
                    origin: aggregate_metric_dicts(repeats)
                    for origin, repeats in method_data["origins"].items()
                    if repeats
                },
                "repeats": method_data["repeats"],
            }
        output_scales.append({
            "requested_bank_size": scale["requested_bank_size"],
            "bank_size": scale["bank_size"],
            "methods": methods,
            "diagnostics": {
                method: examples[:DIAGNOSTIC_LIMIT]
                for method, examples in scale["diagnostics"].items()
            },
        })
    return {
        "cohort_fingerprint": cohort_fingerprint(cohort_payload),
        "scales": output_scales,
    }


def calculate_causality(
    experiment_mode: str,
    dataset_source: str,
    overwrite_output: bool,
) -> None:
    experiment = experiment_configuration(experiment_mode)
    dataset = load_causality_dataset(dataset_source)
    records = build_causality_records(dataset, RECORD_LIMIT)
    dataset_metadata = causality_dataset_metadata(dataset, records, dataset_source)
    output_path = result_output_path(experiment_mode)
    if output_path.exists() and not overwrite_output:
        if result_matches_configuration(output_path, experiment, dataset_metadata):
            print(f"skipped_existing={output_path}")
            return
        print(f"recalculating_stale={output_path}")

    all_queries = flatten_queries(records)
    if experiment_mode == "local":
        queries = sample_queries(
            all_queries,
            LOCAL_QUERY_LIMIT,
            RANDOM_SEED,
            "local-query-limit",
        )
        global_cohorts = None
        used_queries = queries
    else:
        queries = None
        global_cohorts = build_global_cohorts(
            records,
            GLOBAL_BANK_SIZES,
            GLOBAL_REPEATS,
            GLOBAL_QUERY_OWNER_COUNT,
            GLOBAL_QUERY_LIMIT,
            RANDOM_SEED,
        )
        used_query_by_id = {
            query["query_id"]: query
            for cohort in global_cohorts
            for query in cohort["queries"]
        }
        used_queries = list(used_query_by_id.values())
    edits = [record["edit"] for record in records]
    bm25 = Bm25Index([edit["prompt"] for edit in edits])
    model = load_embedding_model()
    try:
        edit_prompt_embeddings = encode_prompts(
            model, [edit["prompt"] for edit in edits]
        )
        subject_embeddings = encode_prompts(
            model, [edit["subject"] for edit in edits]
        )
        composed_embeddings = encode_prompts(
            model, [composed_edit_text(edit) for edit in edits]
        )
        query_embeddings = encode_prompts(
            model, [query["prompt"] for query in used_queries]
        )
        query_embedding_by_id = {
            query["query_id"]: query_embeddings[index]
            for index, query in enumerate(used_queries)
        }
        if experiment_mode == "local":
            analysis = run_local_experiment(
                records,
                queries,
                bm25,
                query_embedding_by_id,
                edit_prompt_embeddings,
                subject_embeddings,
                composed_embeddings,
            )
        else:
            analysis = run_global_experiment(
                records,
                bm25,
                global_cohorts,
                query_embedding_by_id,
                edit_prompt_embeddings,
                subject_embeddings,
                composed_embeddings,
            )
    finally:
        release_embedding_model(model)

    result = {
        "embedding_model": EMBEDDING_MODEL_REPO,
        "representation": "normalized_dense_sentence_embedding",
        "use_fp16": EMBEDDING_USE_FP16,
        "compute_dtype": embedding_compute_dtype(),
        "max_length": EMBEDDING_MAX_LENGTH,
        "dataset": dataset_metadata,
        "experiment": experiment,
        "analysis": analysis,
    }
    write_json_atomic(output_path, result)
    print(f"saved={output_path}")

## Tables And Plots

In [ ]:
def load_result(experiment_mode: str) -> dict:
    path = result_output_path(experiment_mode)
    if not path.exists():
        raise FileNotFoundError(f"Calculate the experiment first: {path}")
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def local_metric_table(result: dict) -> pd.DataFrame:
    rows = []
    for method in SCORER_ORDER:
        for cohort, cohort_data in result["analysis"]["methods"][method]["cohorts"].items():
            for group, metrics in cohort_data["metrics"].items():
                rows.append({
                    "Scorer": SCORER_LABELS[method],
                    "Negative cohort": cohort,
                    "Group": group,
                    "Queries": metrics.get("count", 0),
                    "AUC": metrics.get("pairwise_auc"),
                    "Top-1": metrics.get("top1_accuracy"),
                    "MRR": metrics.get("mrr"),
                    "Best-negative gap": metrics.get("mean_best_negative_gap"),
                })
    return pd.DataFrame(rows).set_index(["Scorer", "Negative cohort", "Group"])


def local_threshold_table(result: dict) -> pd.DataFrame:
    rows = []
    for method in SCORER_ORDER:
        for cohort, cohort_data in result["analysis"]["methods"][method]["cohorts"].items():
            threshold = cohort_data["threshold"]
            rows.append({
                "Scorer": SCORER_LABELS[method],
                "Negative cohort": cohort,
                "Available": threshold.get("available", False),
                "Threshold": threshold.get("threshold"),
                "Positive recall": threshold.get("evaluation_positive_recall"),
                "Negative FPR": threshold.get("evaluation_negative_false_positive_rate"),
                "Precision": threshold.get("evaluation_precision"),
            })
    return pd.DataFrame(rows).set_index(["Scorer", "Negative cohort"])


def plot_local_metrics(result: dict) -> None:
    metrics = (
        ("pairwise_auc", "Pairwise AUC"),
        ("top1_accuracy", "Strict top-1"),
        ("mrr", "MRR"),
        ("mean_best_negative_gap", "Best-negative gap"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    x = np.arange(len(SCORER_ORDER))
    width = 0.36
    for axis, (metric_key, title) in zip(axes.flat, metrics):
        for offset, cohort in ((-width / 2, "random"), (width / 2, "challenge")):
            values = [
                result["analysis"]["methods"][method]["cohorts"][cohort]["metrics"]["all"][metric_key]
                for method in SCORER_ORDER
            ]
            axis.bar(x + offset, values, width=width, label=cohort)
        axis.set_title(title)
        axis.set_xticks(x, [SCORER_LABELS[method] for method in SCORER_ORDER], rotation=18, ha="right")
        axis.grid(axis="y", alpha=0.25)
        if metric_key == "mean_best_negative_gap":
            axis.axhline(0.0, color="black", linewidth=0.8)
    axes[0, 0].legend()
    fig.tight_layout()
    plt.show()


def global_metric_table(result: dict) -> pd.DataFrame:
    rows = []
    for scale in result["analysis"]["scales"]:
        for method in SCORER_ORDER:
            metrics = scale["methods"][method]["aggregate"]
            rows.append({
                "Scorer": SCORER_LABELS[method],
                "Bank size": scale["bank_size"],
                "R@1": metrics["recall_at_1"]["mean"],
                "R@5": metrics["recall_at_5"]["mean"],
                "R@10": metrics["recall_at_10"]["mean"],
                "R@20": metrics["recall_at_20"]["mean"],
                "R@50": metrics["recall_at_50"]["mean"],
                "MRR": metrics["mrr"]["mean"],
                "Rank p50": metrics["rank_p50"]["mean"],
                "Rank p95": metrics["rank_p95"]["mean"],
            })
    return pd.DataFrame(rows).set_index(["Scorer", "Bank size"])


def global_position_table(result: dict) -> pd.DataFrame:
    rows = []
    for scale in result["analysis"]["scales"]:
        for method in SCORER_ORDER:
            for position, metrics in scale["methods"][method]["positions"].items():
                rows.append({
                    "Scorer": SCORER_LABELS[method],
                    "Bank size": scale["bank_size"],
                    "Source position": int(position),
                    "R@1": metrics["recall_at_1"]["mean"],
                    "R@10": metrics["recall_at_10"]["mean"],
                    "MRR": metrics["mrr"]["mean"],
                    "Rank p50": metrics["rank_p50"]["mean"],
                })
    return pd.DataFrame(rows).set_index(["Scorer", "Bank size", "Source position"])


def global_origin_table(result: dict) -> pd.DataFrame:
    rows = []
    for scale in result["analysis"]["scales"]:
        for method in SCORER_ORDER:
            for origin, metrics in scale["methods"][method]["origins"].items():
                rows.append({
                    "Scorer": SCORER_LABELS[method],
                    "Bank size": scale["bank_size"],
                    "Origin": origin,
                    "R@1": metrics["recall_at_1"]["mean"],
                    "R@10": metrics["recall_at_10"]["mean"],
                    "MRR": metrics["mrr"]["mean"],
                    "Rank p50": metrics["rank_p50"]["mean"],
                })
    return pd.DataFrame(rows).set_index(["Scorer", "Bank size", "Origin"])


def plot_global_metrics(result: dict) -> None:
    metrics = (
        ("recall_at_1", "Recall@1"),
        ("recall_at_10", "Recall@10"),
        ("mrr", "MRR"),
        ("rank_p50", "Median rank"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for axis, (metric_key, title) in zip(axes.flat, metrics):
        for method in SCORER_ORDER:
            sizes = [scale["bank_size"] for scale in result["analysis"]["scales"]]
            means = [
                scale["methods"][method]["aggregate"][metric_key]["mean"]
                for scale in result["analysis"]["scales"]
            ]
            axis.plot(sizes, means, marker="o", label=SCORER_LABELS[method])
        axis.set_xscale("log")
        axis.set_title(title)
        axis.set_xlabel("Edit bank size")
        axis.grid(alpha=0.25)
        if metric_key in {"recall_at_1", "recall_at_10", "mrr"}:
            axis.set_ylim(0.0, 1.0)
        if metric_key == "rank_p50":
            axis.set_ylabel("Lower is better")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def diagnostic_table(result: dict, experiment_mode: str) -> pd.DataFrame:
    rows = []
    if experiment_mode == "local":
        for method in SCORER_ORDER:
            for cohort, cohort_data in result["analysis"]["methods"][method]["cohorts"].items():
                for example in cohort_data["diagnostics"]:
                    rows.append({
                        "Scorer": SCORER_LABELS[method],
                        "Setting": cohort,
                        "Position": example["source_edit_position"],
                        "Query": example["query"],
                        "Expected edit": example["expected_edit"]["prompt"],
                        "Retrieved edit": example["retrieved_edit"]["prompt"],
                        "Gap": example["gap"],
                    })
    else:
        for scale in result["analysis"]["scales"]:
            for method, examples in scale.get("diagnostics", {}).items():
                for example in examples:
                    rows.append({
                        "Scorer": SCORER_LABELS[method],
                        "Setting": scale["bank_size"],
                        "Position": example["source_edit_position"],
                        "Query": example["query"],
                        "Expected edit": example["expected_edit"]["prompt"],
                        "Retrieved edit": example["retrieved_edit"]["prompt"],
                        "Gap": example["gap"],
                    })
    return pd.DataFrame(rows)


def visualize_causality(experiment_mode: str) -> None:
    result = load_result(experiment_mode)
    if result["experiment"] != experiment_configuration(experiment_mode):
        raise ValueError("Saved result does not match the current experiment configuration.")
    current_dataset = load_causality_dataset(DATASET_SOURCE)
    current_records = build_causality_records(current_dataset, RECORD_LIMIT)
    current_metadata = causality_dataset_metadata(
        current_dataset,
        current_records,
        DATASET_SOURCE,
    )
    if not causality_dataset_metadata_matches(
        result.get("dataset"),
        current_metadata,
    ):
        raise ValueError(
            "Saved causality result does not match the currently loaded dataset "
            "or extraction policy. Recalculate before visualizing."
        )
    print(
        f"dataset_rows={result['dataset']['rows']} "
        f"causal_queries={result['dataset']['queries']} "
        f"cohort_fingerprint={result['analysis']['cohort_fingerprint']}"
    )
    if experiment_mode == "local":
        display(local_metric_table(result))
        display(local_threshold_table(result))
        plot_local_metrics(result)
        print(
            "Interpretation: high local AUC with low global Recall@1 means the scorer "
            "recognizes related edits but cannot select one reliably under competition. "
            "The threshold table is association-only because this dataset contains no "
            "unrelated locality/no-edit prompts."
        )
    else:
        display(global_metric_table(result))
        display(global_position_table(result))
        display(global_origin_table(result))
        plot_global_metrics(result)
        print(
            "Interpretation: stable recall as the retained causality-owner bank grows "
            "supports retrieval-based routing within this selected benchmark. "
            "A position gap shows whether questions that hide the edit subject (position 1) "
            "need a different decomposition or reranking signal."
        )
    diagnostics = diagnostic_table(result, experiment_mode)
    if len(diagnostics):
        display(diagnostics)

## Experiment Configuration

In [ ]:
# Select one-edit association analysis or edit-bank retrieval.
# Options: "local", "global".
EXPERIMENT_MODE = "global"

# Load the causality split from local Arrow or Hugging Face.
# Options: "local", "huggingface".
DATASET_SOURCE = "local"

# Embedding inference device, precision, token limit, and batch size.
EMBEDDING_GPU_ID = 0
EMBEDDING_USE_FP16 = True
EMBEDDING_MAX_LENGTH = 384
EMBEDDING_BATCH_SIZE = 64

# Limit source records only for smoke tests; None uses the complete split.
RECORD_LIMIT = None

# Recalculate the selected result even when its fingerprints still match.
OVERWRITE_EXISTING_RESULT = True

# Shared deterministic cohort seed and bounded saved failures.
RANDOM_SEED = 42
DIAGNOSTIC_LIMIT = 5

# Local mode: one positive plus this many random or challenge negatives.
LOCAL_NEGATIVE_COUNT = 31
LOCAL_QUERY_LIMIT = None
LOCAL_CALIBRATION_FRACTION = 0.5

# Global mode: nested banks add distractors from retained causality owners.
# None means all 17,215 retained owners, not the full merged UniEdit universe.
GLOBAL_BANK_SIZES = [
    100,
    1000,
    2000,
    5000,
    None,
]
GLOBAL_REPEATS = 5
GLOBAL_QUERY_OWNER_COUNT = 100
GLOBAL_QUERY_LIMIT = None

EXPERIMENT = experiment_configuration(EXPERIMENT_MODE)
EXPERIMENT

## Manual Actions

In [ ]:
# Calculate and save the selected local or global experiment:
# calculate_causality(
#     experiment_mode=EXPERIMENT_MODE,
#     dataset_source=DATASET_SOURCE,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
# )

# Display the matching tables, plots, interpretation, and failures:
# visualize_causality(EXPERIMENT_MODE)